# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ntsikelelo-N/Flyrank_ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane 2 — Refresh / Content Opportunity Scoring (predefined, per docs/ml-intern-dataset-and-lane-guide.md).

Why this lane: the part of ML that interests me most is predicting outcomes — "will this page decline?" — and Lane 2 lets me build exactly that model while keeping it attached to a real decision. The lane's question ("which pages should be reviewed first?") is answered by a decline-risk classifier whose scores drive a ranked review queue. The lane guide explicitly points to the stronger label shape I plan to work toward: features from a prior window → decline over a later windo

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Research question: Given limited review capacity, which content pages should a FlyRank content editor review first, based on observable evidence of decline risk and refresh opportunity?

Unit of analysis (grain): one content item (page), with metrics aggregated over a trailing 90-day window. One row = one page.

Output: a ranked review queue — each page gets a refresh-priority score, a suggested action (refresh / expand / protect / monitor), and human-readable reason codes.

Who acts, and what do they do: a content editor or reviewer takes the top K pages from the queue (K set by team capacity, e.g. 20–50 per cycle), inspects each, and decides whether to refresh, expand, protect, prune, or keep monitoring it.

Cost of a wrong call — and the asymmetry:

False positive (page flagged, but healthy): an editor wastes limited review time on a page that didn't need it. Cost = wasted hours; it crowds a truly at-risk page out of the queue.
False negative (declining page missed): the page keeps losing search visibility and traffic until (or unless) it surfaces later. Cost = unrecovered traffic loss, which can compound over months.
Because reviewer time is the scarce resource, the metric that matches the decision is precision@K — of the top K pages surfaced, how many were genuinely worth the look — with recall as the secondary check that we aren't missing expensive problems.

Why data/ML can help at all (why this is not just "train a model"): the baseline alternative is a hand-written rule, and the repo ships one (a weighted score over visibility, freshness risk, position opportunity, and depth gap). A rule like that is the right starting point — but decline risk emerges from many tangled, interacting signals (position, CTR vs. position tier, impression trend, engagement, age, freshness, content depth), and the interactions shift across clients and over time. That is precisely the regime where a learned model can out-rank a fixed rule. The starter pipeline's committed results already show this on this slice: the hand rule's precision@50 is 0.240 vs. 0.740 for a random forest (outputs/model_report.md) — roughly 12 vs. 37 genuinely-positive pages in a reviewer's top 50. My job is not "train a model"; it is to test honestly whether a learned ranking beats a transparent rule under a stronger, future-window label, and to say clearly where it doesn't.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd
from pathlib import Path

# Load the starter dataset: local path first (repo clone), else raw GitHub URL (Colab).
LOCAL = Path("../../data/raw/content_refresh_anonymized.csv")
# URL = ("https://raw.githubusercontent.com/Ntsikelelo-N/Flyrank_ML/"
#        "main/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(LOCAL )
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   Clients: {df['client_id'].nunique()}")

# NUMBER 1 — The base rate: how much of the inventory is "declining" right now?
down = df["trend_direction"].eq("down")
print(f"Pages with trend_direction == 'down': {down.sum():,} of {len(df):,} "
      f"({down.mean()*100:.1f}%)")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

Number 1 — the base rate is 54.2%. More than half the inventory is "declining" by the current-window definition. This single number reframes the whole problem: a binary decline flag is operationally useless — no team can review 16,000+ pages. The real decision is prioritization under capacity, which is exactly what a ranked queue solves. It also motivates a stricter label later: when 54% of pages trip a definition, the definition is casting too wide a net, and a future-window label with magnitude, persistence, and volume requirements should replace it.

In [ ]:
down_with_demand = df[down & (df["impressions_90d"] >= 500)]
print(f"Declining pages with >= 500 impressions_90d: {len(down_with_demand):,}")
print(f"That is {len(down_with_demand)/50:.0f}x a 50-page review capacity.")

Number 2 — even the high-stakes subset overwhelms review capacity 9,961. Restricting to declining pages with at least 500 impressions in 90 days — pages with real demand at stake — still leaves far more candidates than a reviewer's realistic top-50. So even after a sensible volume filter, the queue must be ordered, not just filtered. Ordering well is the entire value of this project.

In [ ]:
has_pos = df["avg_position"] > 0
compare = (
    df[has_pos]
    .assign(declining=down)
    .groupby("declining")[["days_since_last_update", "avg_position",
                           "ctr", "impressions_90d"]]
    .median()
    .rename(index={False: "not_down", True: "down"})
)
print(compare.round(2))

Number 3 — no single signal cleanly separates decliners, median days since last update is 22.0 for pages that are not down and 20.0 for pages that are down. The average position for the pages also is not too far from each other, 11.4 for not down and 11.3 for down.

The median staleness, position, CTR, and volume of declining vs. non-declining pages sit close together — any one-signal threshold would misclassify a large share of pages. The pattern is real but spread across many interacting signals, which is the framing skill's exact criterion for when ML earns its place over an if-statement.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

About the label. The starter label `is_declining_label = (trend_direction == "down")` is a proxy: a bucket computed from the current window (last 30 days vs. the 30 days before), not an observed future outcome. I will treat it as a teaching label only. The capstone target will be future-shaped, features from a prior window → decline over a later window — with an explicit definition covering magnitude, window, persistence, minimum volume, and group checks (consolidation, seasonality, SERP/AI click loss, noise are all decline look-alikes I must rule out, per lane-guide §7).

Leakage rules I commit to now. `trend_direction` and `trend_pct` are the label source and are NEVER features. `content_id/client_id` are pseudonyms for grouping and splits only. Missingness follows `content_type`, so blind fillna(0) silently encodes a category signal. I'll use has-flags instead. Validation will use client-holdout (and time-aware splits once labels are future-shaped).

What a score means. A high refresh score means "review this page first" — it is decision support. It does NOT mean "this page will decline" as a guarantee, and it does NOT mean "a refresh will cause a recovery"; proving causation would need an experiment this data cannot provide. All results are observational: I will say "we observed" and "this suggests", not "this proves". No claims about Google's algorithm, AI citations, or AI rankings, `ai_sessions_90d` only measures click-throughs from AI tools.

Scope. Numbers in this notebook come from a 30k-row anonymized starter slice. Nothing here is a benchmark on the full ~79M-row warehouse; warehouse results must be earned separately with proper validation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.